[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/bloc2_donnees/cours/seance1_cours.ipynb)

# Séance 2.1 — Charger, comprendre et nettoyer un jeu de données

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- dire en 30 secondes ce que contient un fichier que vous n'avez jamais vu
- sélectionner les lignes et les colonnes qui vous intéressent, et calculer sur une colonne entière
- repérer les défauts classiques d'un fichier réel : doublons, trous, types faux
- convertir du texte en nombres et en dates — et vérifier que la conversion dit vrai
- annoncer combien de lignes votre nettoyage a fait perdre, et pourquoi

## Le contexte

Vous venez d'arriver chez un **détaillant en ligne** européen. On vous remet
l'historique des ventes de l'année écoulée et une question simple :

> *« Sur quel marché faut-il investir l'an prochain ? »*

Vous ne pouvez pas répondre tant que vous ne savez pas ce que contient ce
fichier. Cette séance, c'est exactement ça : **prendre en main un jeu de
données qu'on n'a jamais vu**, puis le **remettre en état** — parce qu'un
fichier réel n'est jamais propre.

Nous avons trois fichiers :

| Fichier | Une ligne = | Colonnes |
|---|---|---|
| `ventes.csv` | un produit dans une commande | `date`, `cmd_id`, `prod_id`, `qte`, `prix`, `client_id` |
| `clients.csv` | un client | `client_id`, `pays`, `segment`, `date_insc` |
| `produits.csv` | un produit | `prod_id`, `libelle`, `categorie` |

## 1. Le point de départ

Cette cellule est présente au début de **tous** les notebooks du cours. Elle
charge les outils dont nous aurons besoin et règle l'affichage pour les petits
écrans. Exécutez-la (bouton ▶) sans chercher à la comprendre pour l'instant.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/Intelligence-Artificielle-et-Data-Science/main/bloc2_donnees/data/"

**`pandas`** est la bibliothèque qui sert à manipuler des tableaux de données
en Python. `pd` est son surnom : on écrira `pd.` chaque fois qu'on l'appelle.

Pensez à pandas comme à **un Excel qu'on pilote par des instructions**. Même
objet — un tableau de lignes et de colonnes — mais au lieu de cliquer, on écrit
ce qu'on veut. L'avantage : ça marche sur 45 000 lignes aussi vite que sur 10,
et on peut relancer exactement la même analyse le mois prochain.

## 2. Charger les données

Une seule ligne. `pd.read_csv()` accepte directement une **adresse web** :
rien à télécharger, rien à ranger dans un dossier.

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")  ## lit le fichier en ligne
ventes.head(3)                             ## les 3 premieres lignes

`ventes` est un **DataFrame** : le nom que pandas donne à un tableau.

`.head(3)` affiche les 3 premières lignes. Toujours commencer par là : c'est
la façon la plus rapide de vérifier que le fichier est bien celui qu'on croit.

> 💡 `head()` prend en argument le **nombre de lignes** à afficher. Sans
> argument, elle en affiche cinq. Dans ce cours nous écrirons souvent
> `head(3)` : trois lignes suffisent presque toujours à vérifier qu'une
> manipulation a marché.

## 3. La carte d'identité d'un fichier

Quatre commandes, toujours dans cet ordre. En 30 secondes vous savez à quoi
vous avez affaire.

In [ ]:
print(ventes.shape)   ## (nombre de lignes, nombre de colonnes)
ventes.info()         ## les colonnes, leur type, les valeurs manquantes

Ce que `info()` vous dit :

- **`45123 entries`** — 45 123 lignes.
- **`non-null`** — combien de valeurs sont renseignées. Ici tout est complet ;
  on verra dans une demi-heure que c'est très rare.
- **`Dtype`** — le *type* de chaque colonne, et c'est le plus important :
  `int64` (nombre entier), `float64` (nombre à virgule), `object` (**du
  texte**).

> ⚠️ Regardez `date` : son type est `object`, c'est-à-dire du **texte**. Pour
> pandas, `"2011-10-04"` est une chaîne de caractères, pas une date. On ne peut
> donc pas encore lui demander « quel mois ? ». C'est un des défauts qu'on
> réparera en seconde partie de séance.

In [ ]:
# 45 123 lignes, mais combien de clients et de commandes distincts ?
print("clients   :", ventes["client_id"].nunique())   ## valeurs distinctes
print("commandes :", ventes["cmd_id"].nunique())      ## idem sur la commande

**Une ligne n'est pas un client.** Une ligne est *un produit dans une
commande*. 45 123 lignes correspondent à 1 955 commandes passées par 472
clients.

C'est la première question à se poser devant n'importe quel fichier :
**une ligne, c'est quoi exactement ?** Se tromper là-dessus, c'est se tromper
sur tout le reste de l'analyse.

## 4. Choisir ce qu'on regarde

### Une colonne, ou plusieurs

Doubles crochets pour plusieurs : les crochets extérieurs veulent dire « je
sélectionne », les intérieurs délimitent la **liste** des colonnes voulues.

In [ ]:
print(ventes["prix"].head(3).tolist())        ## une seule : une Series
ventes[["prod_id", "qte", "prix"]].head(3)    ## une liste -> un tableau

### Position ou étiquette — `.iloc` et `.loc`

`.iloc` raisonne en **positions** : la première ligne, la deuxième — comme on
compterait des lignes à l'écran, à partir de 0. `.loc` raisonne en
**étiquettes** : le nom de la ligne, le nom de la colonne.

In [ ]:
print(ventes.iloc[0]["qte"])      ## la 1re ligne, par sa POSITION
print(ventes.loc[10, "prix"])     ## ligne d'etiquette 10, colonne "prix"

ventes.iloc[0:2, 0:3]             ## avant la virgule les lignes, apres les colonnes

| | `.iloc` | `.loc` |
|---|---|---|
| ce qu'on lui donne | des **positions** : 0, 1, 2... | des **étiquettes** : `10`, `"prix"` |
| `[0:2]` | les positions 0 et 1 — borne haute **exclue** | les étiquettes 0, 1 **et 2** — borne haute **incluse** |

Ici les lignes n'ont pas reçu de nom, alors pandas leur a donné leur numéro
d'arrivée. C'est pour ça que `.loc[0]` et `.iloc[0]` désignent la même ligne —
mais **c'est une coïncidence**, pas une règle : elle ne survit pas au premier
filtrage, où les lignes conservées gardent l'étiquette qu'elles avaient dans
le fichier d'origine.

### Des lignes par une condition — `.query()`

In [ ]:
cheres = ventes.query("prix > 50")   ## la condition, entre guillemets
print(cheres.shape)                  ## combien de lignes ont survecu ?
cheres.head(3)

`.query()` prend une **condition écrite entre guillemets**, presque en français :
`"prix > 50"`, `"qte >= 100"`, `"pays == 'France'"`.

Vous rencontrerez aussi l'autre écriture, plus classique :

```python
ventes[ventes["prix"] > 50]
```

Les deux font exactement la même chose. **Nous utiliserons `.query()` dans ce
cours** : deux fois moins de ponctuation à taper, et beaucoup plus lisible dès
que la condition se complique.

## 5. Résumer en un coup d'œil

In [ ]:
# On selectionne les colonnes AVANT : sinon la sortie deborde de l'ecran
ventes[["qte", "prix"]].describe().round(2)   ## huit statistiques d'un coup

À lire ainsi : `mean` la moyenne (**3,93 €**), `50%` la **médiane** — la valeur
qui coupe la population en deux (**1,95 €**) — et `max` le maximum
(**4 161 €**).

> 📊 Moyenne 3,93 € mais médiane 1,95 € : la moyenne est **deux fois** la
> médiane. C'est la signature d'une poignée de valeurs très élevées qui tirent
> la moyenne vers le haut. Devant un écart pareil, la médiane décrit bien mieux
> « le produit typique ». Un réflexe à garder : **comparer moyenne et médiane
> avant de citer un chiffre en réunion.**

In [ ]:
clients = pd.read_csv(BASE + "clients.csv")

# value_counts() compte les categories, deja trie du plus frequent
print(clients["pays"].value_counts().head(5))
print("prix moyen :", round(ventes["prix"].mean(), 2))

## 6. Calculer sur des colonnes entières

Le chiffre d'affaires d'une ligne, c'est la quantité multipliée par le prix.

In [ ]:
ventes["ca"] = ventes["qte"] * ventes["prix"]  ## 45 123 calculs
ventes[["qte", "prix", "ca"]].head(3)          ## toujours verifier apres

Regardez bien ce qui vient de se passer : **une seule instruction a fait
45 123 multiplications**. On écrit l'opération *une fois, sur la colonne*, et
pandas l'applique à chaque ligne. C'est ce qui rend l'analyse de 45 000 lignes
aussi simple que celle de 10.

Toutes les opérations arithmétiques fonctionnent de cette façon :

| Vous écrivez | Ce que pandas fait |
|---|---|
| `df["a"] + df["b"]` | additionne les deux colonnes, ligne à ligne |
| `df["a"] * 1.2` | multiplie **chaque** ligne par 1,2 |
| `df["a"] / df["a"].sum()` | divise chaque ligne par le total |
| `df["a"].round(2)` | arrondit chaque ligne à 2 décimales |

Deux formes cohabitent, et il faut les distinguer : `ventes["prix"] * 1.2`
applique **le même nombre** à toutes les lignes, tandis que
`ventes["qte"] * ventes["prix"]` fait travailler **deux colonnes ensemble**,
ligne par ligne.

In [ ]:
# Le chiffre d'affaires total de l'annee
print("CA total :", round(ventes["ca"].sum(), 2), "euros")   ## somme colonne

## 7. Apprendre à lire une erreur

Vous allez faire des erreurs en permanence. C'est normal, y compris pour les
professionnels. Ce qui distingue quelqu'un qui avance, c'est qu'il **lit** le
message au lieu de le subir.

La cellule suivante est **volontairement fausse**. Exécutez-la.

In [ ]:
ventes["Prix"]   ## erreur volontaire : la colonne est "prix"

Le message est long et rouge. **Ne lisez que la dernière ligne :**
`KeyError: 'Prix'`, c'est-à-dire *« je n'ai trouvé aucune colonne appelée
`Prix` »*. Python distingue majuscules et minuscules.

| Message | Ce que ça veut dire | Quoi faire |
|---|---|---|
| `KeyError: 'Prix'` | cette colonne n'existe pas | vérifier l'orthographe avec `ventes.columns` |
| `NameError: name 'vente' is not defined` | cette variable n'existe pas | faute de frappe, ou cellule au-dessus pas exécutée |
| `SyntaxError: invalid character '"'` | des guillemets courbes | retaper les guillemets (voir *Bien démarrer*) |

> ⚠️ La troisième est **invisible à l'œil nu** : votre code paraît
> parfaitement correct.

---

## La première moitié de cette séance vous a menti

`ventes.csv` était **impeccable** : pas un trou, pas un doublon, des types
corrects. Ça n'arrive jamais.

Voici le même détaillant, mais l'export tel qu'il sort vraiment du système :
`ventes_sale.csv`.

> 🎯 **Votre mission pour la suite :** transformer ce fichier en données
> exploitables, et savoir dire **combien de lignes** vous avez perdues au
> passage et **pourquoi**.

In [ ]:
sale = pd.read_csv(BASE + "ventes_sale.csv")   ## le fichier brut

print(sale.shape)
sale.head(5)        ## cinq lignes suffisent a reperer l'essentiel

Prenez 30 secondes pour regarder ces cinq lignes. Qu'est-ce qui cloche ?

In [ ]:
sale.info()   ## regarder surtout la colonne Dtype

### Le diagnostic

`info()` révèle déjà trois problèmes :

- **`prix` est de type `object`** — c'est du **texte**, pas un nombre. Les
  coupables sont visibles dès les cinq premières lignes : le suffixe de
  `0,42 EUR`, et la **virgule** décimale là où Python attend un point.
- **`date` est de type `object`** — du texte aussi. Impossible de demander
  « quel mois ? ».
- **`client_id` a des trous**, visible sur le `non-null`.

Il y en a trois autres qu'`info()` ne montre pas. On va les débusquer.

## Défaut 1 — Les doublons

**On commence toujours par là**, avant toute autre étape de nettoyage.

La raison est un problème de comptage. Si vous commencez par retirer les ventes
sans client, vous noterez « 407 lignes retirées ». Mais parmi ces 407,
certaines étaient des copies l'une de l'autre : vous n'avez donc pas retiré 407
ventes, et vous ne saurez jamais combien. En dédoublonnant d'abord, chaque ligne
est une vente distincte, et tous les comptes qui suivent veulent dire quelque
chose.

In [ ]:
# duplicated() marque True chaque ligne deja vue plus haut
print("lignes strictement identiques :", sale.duplicated().sum())

avant = len(sale)                        ## on note le point de depart
propre = sale.drop_duplicates().copy()   ## .copy() : un vrai tableau a soi
print(avant, "->", len(propre))          ## toujours mesurer ce qu'on retire

251 lignes en double. Un client ne passe pas deux fois exactement la même
commande, à la même seconde, pour le même produit : c'est un **bug d'export**.

> ⚠️ Ici les lignes sont **strictement identiques sur toutes les colonnes**,
> donc on peut supprimer. Si seul le `cmd_id` était en double, ce pourrait être
> un vrai client commandant deux fois le même article. Vérifiez toujours **sur
> quelles colonnes** porte le doublon avant de supprimer.

> 💡 Le `.copy()` dit à pandas : « fais-moi un vrai tableau indépendant ».
> Sans lui, il vous avertira plus tard que vous modifiez peut-être une simple
> vue du tableau d'origine. Prenez l'habitude de l'ajouter après un filtrage.

## Défaut 2 — Les valeurs manquantes

In [ ]:
propre.isna().sum()   ## un compte de trous, colonne par colonne

407 lignes sans `client_id`. **Que faire ?**

Il n'y a pas de réponse universelle. Il y a une question à se poser :
**pourquoi cette valeur manque-t-elle ?** Ici, probablement des ventes sans
compte client (achat en magasin, commande invitée). Donc :

| Votre question | La bonne décision |
|---|---|
| « Combien mes clients dépensent-ils ? » | **Supprimer** ces lignes : elles n'ont pas de client |
| « Quel est mon chiffre d'affaires total ? » | **Les garder** : ce sont de vraies ventes |

### À quoi ressemblerait `fillna`, concrètement

In [ ]:
# On ecrit dans une colonne A COTE, jamais par-dessus l'originale
propre["client_id_new"] = propre["client_id"].fillna(0)   ## 0 dans les trous

print("trous restants :", propre["client_id_new"].isna().sum())
print(propre["client_id_new"].value_counts().head(3))

Plus un seul trou : mission accomplie ? Regardez le classement. Le « client
0 » arrive **deuxième du fichier** avec 407 achats, derrière un seul client
réel. Sauf que ce client n'existe pas : ce sont 407 acheteurs différents
regroupés sous une étiquette inventée. Toute analyse par client sera fausse,
et absolument rien ne vous préviendra.

> ⚠️ **Le piège à ne jamais commettre :** `fillna(0)` sur un identifiant.
> Remplir une valeur manquante, c'est **inventer une donnée** — ne le faites
> que si vous pouvez le justifier.

`fillna` a pourtant des usages légitimes : une quantité absente qu'on sait
valoir 0, un libellé vide qu'on remplace par `"inconnu"`. La question n'est
jamais « est-ce que ça marche ? » mais « qu'est-ce que j'affirme en remplissant
ce trou ? ».

In [ ]:
# Celle-la ne nous sert a rien : on la retire avant de continuer
propre = propre.drop(columns=["client_id_new"])   ## drop(columns=[...])

# Notre question portera sur les clients : on supprime ces lignes,
# mais on note combien on en perd.
avant = len(propre)
propre = propre.dropna(subset=["client_id"]).copy()   ## cette colonne seule
print(avant, "->", len(propre), f"({avant - len(propre)} lignes retirees)")

## Défaut 3 — Des nombres stockés en texte

C'est le défaut le plus courant, et le plus sournois. Vous l'avez déjà croisé
au bloc 1 : `2 * "50"` donne `"5050"`, sans le moindre message d'erreur. Ici,
c'est la même chose sur 5 000 lignes.

In [ ]:
propre["prix"].head(4)   ## du texte, pas des nombres

Deux problèmes dans une seule colonne : le suffixe **` EUR`**, et la
**virgule** décimale — convention française, alors que Python attend un point.

Pour réparer, il faut manipuler du texte. pandas range ses outils de texte
derrière **`.str`**, et là encore toute la colonne est traitée d'un coup :

| Commande | Effet |
|---|---|
| `.str.lower()` / `.str.upper()` | tout en minuscules / en majuscules |
| `.str.strip()` | enlève les espaces au début et à la fin |
| `.str.replace("a", "b")` | remplace un morceau de texte |
| `.str.contains("Pays")` | vrai si la chaîne contient ce texte |
| `.str[-4:]` | découpe par position — ici les 4 derniers caractères |

Trois étapes, dans cet ordre :

In [ ]:
# .str donne acces aux operations sur du texte, colonne entiere d'un coup
prix_txt = propre["prix"].str.replace(" EUR", "", regex=False)   ## 1. l'unite
prix_txt = prix_txt.str.replace(",", ".", regex=False)           ## 2. la virgule

propre["prix"] = pd.to_numeric(prix_txt, errors="coerce")   ## 3. la conversion

# Reflexe obligatoire apres un coerce : combien de valeurs ont ete perdues ?
print(propre["prix"].dtype, "|", propre["prix"].isna().sum(), "non convertis")

**`errors="coerce"`** veut dire : *« si tu n'arrives pas à convertir une
valeur, mets `NaN` au lieu de tout faire planter »*. C'est très pratique — et
très dangereux si on ne vérifie pas ensuite.

Zéro perte : notre conversion est propre. **Faites systématiquement cette
vérification.** Sans elle, vous pourriez transformer silencieusement 3 000 prix
en `NaN` et ne vous en apercevoir qu'en présentant vos résultats.

## Défaut 4 — Les dates

Le plus piégeux. On procède en trois temps — mais d'abord, une question de
méthode.

### Temps 0 : savoir ce qu'on attend

On ne peut pas juger si une conversion a réussi sans savoir à quoi devrait
ressembler le résultat. La colonne est encore du texte, mais du texte
régulier : `24/11/2011`, `24-11-2011`. Ses **quatre derniers caractères** sont
donc l'année, quel que soit le séparateur.

In [ ]:
annee = propre["date"].str[-4:]    ## les 4 derniers caracteres

print(annee.value_counts().to_dict())

Deux années : 335 lignes en 2010 et 4 377 en 2011. Ce fichier couvre l'année
écoulée, qui se termine fin 2011. Retenez ce repère : c'est lui qui va nous
permettre, dans deux cellules, de repérer une conversion qui a échoué sans le
dire.

**Temps 1 :** la façon naïve.

In [ ]:
# Cellule volontairement fausse : lisez le message d'erreur
pd.to_datetime(propre["date"])   ## sans format, pandas devine... et echoue

`ValueError: time data "24-11-2011" doesn't match format "%d/%m/%Y"`

Le fichier mélange deux écritures : `14/11/2011` et `24-11-2011`. pandas veut
un format unique. Le message suggère lui-même la solution : `format="mixed"`.

**Temps 2 :** on ajoute `format="mixed"`.

In [ ]:
essai = pd.to_datetime(propre["date"], format="mixed")   ## deux ecritures

print("date la plus ancienne :", essai.min())
print("date la plus recente  :", essai.max())

Plus d'erreur. Mais confrontez le résultat au repère du temps 0 : le fichier
couvre 2010 et 2011, et pandas annonce **janvier 2010** comme date la plus
ancienne. **C'est faux** — les 335 lignes de 2010 sont des lignes de décembre.

Pourquoi ? Parce que `01/12/2010` a été lu **à l'américaine** : mois d'abord,
donc le 12 janvier. En français, c'est le 1er décembre.

**Temps 3 :** on impose la lecture française avec `dayfirst=True`.

In [ ]:
# dayfirst=True : lecture francaise, le jour avant le mois
propre["date"] = pd.to_datetime(propre["date"], format="mixed", dayfirst=True)

print("date la plus ancienne :", propre["date"].min())
print("date la plus recente  :", propre["date"].max())

> ⚠️ **Le point le plus important de la séance.** L'étape 1 produisait une
> **erreur bruyante** : gênante, mais elle vous arrête. L'étape 2 produisait
> une **erreur silencieuse** : le code tourne, les chiffres s'affichent, et
> ils sont faux. C'est de très loin la plus dangereuse.
>
> Après toute conversion, **vérifiez que le résultat est plausible** :
> `.min()`, `.max()`, un `head()`. Trente secondes qui vous éviteront de
> présenter des chiffres faux.

Une fois la colonne convertie en date, `.dt` ouvre tout :

In [ ]:
propre["mois"] = propre["date"].dt.month           ## .dt = boite a outils
propre["jour_sem"] = propre["date"].dt.dayofweek   ## 0 = lundi, 6 = dimanche

propre[["date", "mois", "jour_sem"]].head(3)

## Défaut 5 — Du texte incohérent

In [ ]:
print("categories distinctes :", propre["categorie"].nunique())
print(propre["categorie"].unique()[:8])

24 catégories, alors qu'il n'en existe que 8. Regardez bien : `' cuisine'`
avec un espace devant, `'CUISINE'` en majuscules, `'cuisine'`. Pour pandas,
ce sont **trois catégories différentes** — et un `value_counts()` sur cette
colonne éclaterait la cuisine en trois lignes, chacune sous-estimée.

> ⚠️ L'espace en début de chaîne est **invisible à l'écran**. C'est ce qui
> rend ce défaut particulièrement traître.

In [ ]:
# .str.strip() enleve les espaces au bord, .str.lower() met en minuscules
propre["categorie"] = propre["categorie"].str.strip().str.lower()   ## 24 -> 8

print("apres nettoyage :", propre["categorie"].nunique(), "categories")

## Défaut 6 — Les valeurs aberrantes

In [ ]:
propre["qte"].describe().round(1)   ## regarder min et max avant tout

Un minimum **négatif** et un maximum à **99 999**. Deux anomalies, mais elles
n'ont rien à voir :

- **`qte` négatif** : ce sont des **retours**. Ce n'est pas une erreur, c'est
  une information métier. On les écarte du calcul de chiffre d'affaires, mais
  on ne les jette pas — un taux de retour, ça s'analyse.
- **`qte = 99999`** : personne ne commande 99 999 articles. C'est une saisie
  erronée, ou un code sentinelle. On l'écarte.

> Traiter ces deux cas de la même façon serait une faute d'analyse.

In [ ]:
print("retours :", len(propre.query("qte < 0")), "lignes")
print("quantites aberrantes :", len(propre.query("qte >= 10000")), "lignes")

avant = len(propre)
propre = propre.query("qte > 0 and qte < 10000").copy()   ## "and" dans query
print(avant, "->", len(propre))

## Le bilan

Bonne pratique pour finir : un petit compte rendu de ce qu'on a retiré. Ça
tient en quatre lignes, et ça évite d'avoir à se demander, trois semaines plus
tard, d'où viennent les lignes qui manquent.

In [ ]:
print("lignes au depart  :", len(sale))
print("lignes conservees :", len(propre))
print("taux de perte     :", round(100 * (1 - len(propre) / len(sale)), 1), "%")
print("dont : 251 doublons, 407 sans client, 107 retours, 14 aberrantes")

14,5 % de pertes, et chacune s'explique. C'est plus utile à un lecteur que
« j'ai nettoyé les données ».

Le chiffre sert aussi de garde-fou : à 40 % de pertes, il vaudrait mieux
reprendre le pipeline depuis le début.

**La séance 2.2 repart du fichier propre** et pose enfin la question du début :
sur quel marché faut-il investir ?

---

## Ce que vous savez faire maintenant

### Comprendre un fichier

| Vous voulez... | La commande |
|---|---|
| charger un fichier | `pd.read_csv(url)` |
| voir les premières lignes | `df.head(3)` |
| connaître la taille | `df.shape` |
| voir les colonnes et leurs types | `df.info()` |
| compter les valeurs distinctes | `df["client_id"].nunique()` |
| une colonne / plusieurs | `df["prix"]` / `df[["prix", "qte"]]` |
| une ligne par sa position | `df.iloc[0]` |
| une case par son étiquette | `df.loc[10, "prix"]` |
| des lignes par condition | `df.query("prix > 10")` |
| résumé chiffré | `df[["qte", "prix"]].describe()` |
| compter les catégories | `df["pays"].value_counts()` |
| moyenne, médiane, total, max | `.mean()`, `.median()`, `.sum()`, `.max()` |
| créer une colonne | `df["ca"] = df["qte"] * df["prix"]` |

### Nettoyer un fichier

| Le problème | La commande |
|---|---|
| compter les doublons | `df.duplicated().sum()` |
| supprimer les doublons | `df.drop_duplicates()` |
| repérer les manquants | `df.isna().sum()` |
| supprimer les lignes incomplètes | `df.dropna(subset=["client_id"])` |
| remplacer les manquants | `df["prix"].fillna(0)` |
| texte → nombre | `pd.to_numeric(col, errors="coerce")` |
| texte → date | `pd.to_datetime(col, format="mixed", dayfirst=True)` |
| extraire le mois | `df["date"].dt.month` |
| nettoyer du texte | `col.str.strip().str.lower()` |
| enlever un morceau de texte | `col.str.replace(" EUR", "")` |

## Les trois réflexes à emporter

1. **Devant un fichier inconnu, toujours dans cet ordre :** `shape`, `info()`,
   `head(3)`, `describe()`. Quatre commandes, et vous savez à quoi vous avez
   affaire. Et la première question à se poser : **une ligne, c'est quoi ?**

2. **Le nettoyage est un pipeline, pas une série de bricolages.** Écrivez-le
   dans l'ordre, de haut en bas, en repartant toujours du fichier brut. Le
   jour où on vous livre le fichier du mois suivant, vous relancez le notebook
   et c'est fini.

3. **Notez toujours combien de lignes vous perdez à chaque étape.** Un
   nettoyage qui fait disparaître 40 % des données n'est pas un nettoyage,
   c'est une erreur.